# Transform every channel in one imaging round

Registration is usually estimated from one structural channel. This notebook reuses that round's optional crop geometry, affine transform, and nonlinear warp for the remaining intensity channels.

In [ ]:
# EDIT THESE PATHS and naming rules before running the notebook.
from pathlib import Path

DATA_DIRECTORY = Path("path/to/unregistered_channels")
TRANSFORM_DIRECTORY = Path("path/to/registration_output")
OUTPUT_DIRECTORY = Path("path/to/registered_channels")
FIXED_IMAGE = Path("path/to/original_fixed_registration_channel.tif")
ROUND_NUMBER = 2
FIXED_CROP_INDICES: Path | None = (
    TRANSFORM_DIRECTORY / f"fixed_crop_indices_{ROUND_NUMBER}.json"
)
MOVING_CROP_INDICES: Path | None = (
    TRANSFORM_DIRECTORY / f"moving_crop_indices_{ROUND_NUMBER}.json"
)
# Set both crop paths to None when registration used crop=False.
CHANNEL_PATTERN = f"round{ROUND_NUMBER}_*.tif"
PIXEL_SIZE_XYZ = (0.15, 0.15, 0.15)
ITK_THREADS = 1
BLAS_THREADS = 1

For a `crop=True` registration, keep both crop paths set. The exclusive NumPy `(z0, z1, y0, y1, x0, x1)` bounds are rebuilt as physical ANTs images with the same origin calculation used during registration. For `crop=False`, set both paths to `None`. Public pixel size remains `(x, y, z)` in micrometers.

In [ ]:
import importlib
import json

from cyexm.io.images import crop_array, load_ants_image
from cyexm.io.tiff import write_uint16_tiff
from cyexm.registration.transforms import transform_image
from cyexm.runtime import NativeThreadConfig, configure_native_threads

NATIVE_THREADS = NativeThreadConfig(ITK_THREADS, BLAS_THREADS)
configure_native_threads(NATIVE_THREADS)
ants = importlib.import_module("ants")

AFFINE_TRANSFORM = TRANSFORM_DIRECTORY / f"linear_transform_{ROUND_NUMBER}.mat"
NONLINEAR_WARP = TRANSFORM_DIRECTORY / f"nonlinear_warp_{ROUND_NUMBER}.nii.gz"


def read_bounds(path):
    return None if path is None else tuple(json.loads(path.read_text(encoding="utf-8")))


fixed_bounds = read_bounds(FIXED_CROP_INDICES)
moving_bounds = read_bounds(MOVING_CROP_INDICES)
if (fixed_bounds is None) != (moving_bounds is None):
    raise ValueError("Set both crop paths or set both to None")


def crop_spatial_image(image, bounds):
    if bounds is None:
        return image
    z0, _z1, y0, _y1, x0, _x1 = bounds
    data = crop_array(image.numpy(), bounds)
    origin = ants.transform_index_to_physical_point(image, (z0, y0, x0))
    return ants.from_numpy(
        data,
        origin=origin,
        spacing=image.spacing,
        direction=image.direction,
    )


fixed_full = load_ants_image(FIXED_IMAGE, PIXEL_SIZE_XYZ)
fixed = crop_spatial_image(fixed_full, fixed_bounds)
channel_paths = sorted(DATA_DIRECTORY.glob(CHANNEL_PATTERN))
if not channel_paths:
    raise FileNotFoundError(
        f"No channels match {CHANNEL_PATTERN!r} in {DATA_DIRECTORY}"
    )
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

In [ ]:
for channel_path in channel_paths:
    moving_full = load_ants_image(channel_path, PIXEL_SIZE_XYZ)
    moving = crop_spatial_image(moving_full, moving_bounds)
    registered = transform_image(
        fixed=fixed,
        moving=moving,
        affine_path=AFFINE_TRANSFORM,
        warp_path=NONLINEAR_WARP,
        interpolation="linear",
        native_threads=NATIVE_THREADS,
    )
    output_path = OUTPUT_DIRECTORY / channel_path.name
    write_uint16_tiff(
        registered,
        output_path,
        float(moving.numpy().min()),
        float(moving.numpy().max()),
    )
    print(f"Wrote {output_path}")